 의사 & 환자 대화 데이터셋을 불러와서 해서 텍스트 요약

In [5]:
!pip -q install -U openai pandas openpyxl tqdm

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [ ]:
openai.api_key = "YOUR_OPENAI_API_KEY"

In [6]:
import os
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# API Key 설정 (둘 중 하나)
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

client = OpenAI()  # OPENAI_API_KEY 사용 :contentReference[oaicite:1]{index=1}

FILE_PATH = "/content/medical_conversation_HealthCareMagic_sample (1).xlsx"
SHEET_NAME = 0

CONV_COL = "id"
ORDER_COL = "turn_index"
SPEAKER_COL = "from"   # doctor/client 들어있는 컬럼
TEXT_COL = "text"

MODEL = "gpt-4o-mini"  # 또는 다른 모델로 변경 가능

MAX_CONVS = None       # 예: 5로 두면 5개 대화만 출력

# -----------------------
# 데이터 로드/정렬
# -----------------------
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
print("Columns:", list(df.columns))

df[SPEAKER_COL] = df[SPEAKER_COL].astype(str).str.strip().str.lower()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df.sort_values([CONV_COL, ORDER_COL]) if ORDER_COL in df.columns else df.sort_values([CONV_COL])

print("\n[from 값 분포]")
print(df[SPEAKER_COL].value_counts())

# -----------------------
# 긴 텍스트 대비 청크 요약
# -----------------------
def chunk_text(text: str, chunk_chars: int = 2500, overlap: int = 200):
    text = (text or "").strip()
    if not text:
        return []
    if len(text) <= chunk_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + chunk_chars)
        part = text[start:end].strip()
        if part:
            chunks.append(part)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

def call_llm(system_prompt: str, user_text: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text},
        ],
        temperature=0.2,
        max_output_tokens=350,
        store=False,
    )
    return (resp.output_text or "").strip()  # :contentReference[oaicite:2]{index=2}

def summarize_role_text(text: str, role_label: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""

    if role_label == "doctor":
        system_prompt = (
            "너는 의료 상담 대화 요약가다. 입력은 doctor 발화만 모은 텍스트다. "
            "한국어로 3~6문장 요약. 사실 기반으로만. "
            "포함: (1) 의사의 핵심 판단/설명 (2) 권고/처치/주의사항 (3) 다음 단계."
        )
    else:
        system_prompt = (
            "너는 의료 상담 대화 요약가다. 입력은 client(환자) 발화만 모은 텍스트다. "
            "한국어로 3~6문장 요약. 사실 기반으로만. "
            "포함: (1) 증상/기간/강도 (2) 동반증상/기저질환/복용약(있으면) (3) 질문/요청사항."
        )

    chunks = chunk_text(text)
    if len(chunks) == 1:
        return call_llm(system_prompt, chunks[0])

    partials = [call_llm(system_prompt, ch) for ch in chunks]
    merged = "\n".join(partials).strip()
    return call_llm(system_prompt, merged)

# -----------------------
# id별 doctor/client 분리 후 요약 출력
# -----------------------
grouped = df.groupby(CONV_COL, sort=False)
to_print = grouped.ngroups if MAX_CONVS is None else min(MAX_CONVS, grouped.ngroups)

print(f"\n대화 요약 출력 시작 (총 {grouped.ngroups}개 중 {to_print}개 출력)\n")

count = 0
for conv_id, g in tqdm(grouped, total=grouped.ngroups, desc="Summarizing conversations"):
    doctor_text = "\n".join(g[g[SPEAKER_COL] == "doctor"][TEXT_COL].tolist()).strip()
    client_text = "\n".join(g[g[SPEAKER_COL] == "client"][TEXT_COL].tolist()).strip()

    doctor_summary = summarize_role_text(doctor_text, "doctor")
    client_summary = summarize_role_text(client_text, "client")

    print("=" * 80)
    print(f"Conversation id: {conv_id}")
    print("-" * 80)
    print("[Client Summary]")
    print(client_summary if client_summary else "(client 발화 없음)")
    print("\n[Doctor Summary]")
    print(doctor_summary if doctor_summary else "(doctor 발화 없음)")

    count += 1
    if MAX_CONVS is not None and count >= MAX_CONVS:
        break


Columns: ['id', 'turn_index', 'from', 'text']

[from 값 분포]
from
client    8
doctor    8
Name: count, dtype: int64

대화 요약 출력 시작 (총 1개 중 1개 출력)



Summarizing conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Conversation id: 0
--------------------------------------------------------------------------------
[Client Summary]
환자는 오늘 아침부터 의자에 앉아 있을 때 방이 돌아가는 듯한 어지럼증을 느꼈으며, 움직일 때 증상이 심해졌고 집중하기 어려웠으며 메스꺼움도 동반되었습니다. 구토를 시도했으나 실패했고, 화장실 이용 후 증상이 다소 완화되었습니다. 환자는 오늘 아침 처음으로 이러한 증상을 경험했으며, 베타히스틴 복용과 전정 재활 운동을 시도하고 이비인후과 전문의 진료를 요청했습니다.

[Doctor Summary]
환자의 증상은 양성돌발성 위치성 현훈(BPPV)으로 판단됩니다. 이 질환은 귀 문제로 인해 발생하며, 일반적으로 며칠 내에 호전될 수 있습니다. 의사는 베타히스틴 알약 복용과 전정 재활 운동을 권장하며, 이비인후과 전문의의 진료도 필요하다고 설명했습니다. 다음 단계로는 이비인후과 전문의의 진료 일정을 예약하고 처방전을 받는 것입니다. 추가 질문이 있는지 확인했습니다.
